# Verify routing path

Deploy and exercise the 53/47 PTU routing lab, inspect its telemetry, and delete all resources when finished.

> **Cost warning:** this lab creates three 15-PTU provisioned deployments. Provisioned capacity is billed while deployed, even when idle. Run the cleanup cell as soon as testing is complete.

## 1. Configuration and helpers

Set `subscription_id` if the desired subscription is not already selected by Azure CLI. After a kernel restart, run this cell and then **4. Send traffic**; the traffic cell reloads the deployment details itself.

In [ ]:
subscription_id = ""  # Optional: Azure subscription ID or name
deployment_name = "verify-routing-path"

resource_group_name = f"lab-{deployment_name}"
resource_group_location = "eastus2"

print(f'Resource Group: {resource_group_name}')

# Cleanup remains disabled until this exactly matches resource_group_name.
#cleanup_confirm = ""
cleanup_confirm = resource_group_name

# --- Helpers: nothing below needs editing ---
import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path

def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "labs" / "verify-routing-path" / "main.bicep").is_file():
            return candidate
    raise RuntimeError("Run this notebook from the repository or its labs/verify-routing-path folder.")

repo_root = find_repo_root()
lab_dir = repo_root / "labs" / "verify-routing-path"
template_file = lab_dir / "main.bicep"
parameters_file = lab_dir / "params.json"
invoke_script = lab_dir / "invoke-lab.ps1"
telemetry_file = lab_dir / "telemetry.kql"
powershell = shutil.which("pwsh") or shutil.which("powershell")
if not powershell:
    raise RuntimeError("PowerShell (pwsh or powershell) was not found on PATH.")

def command_text(command):
    return subprocess.list2cmdline([str(part) for part in command]) if os.name == "nt" else shlex.join([str(part) for part in command])

def run(command, *, display_command=None, capture_output=False, env=None):
    command = [str(part) for part in command]
    shown = [str(part) for part in (display_command or command)]
    print(f"> {command_text(shown)}")
    popen_command = command_text(command) if os.name == "nt" else command
    if capture_output:
        result = subprocess.run(
            popen_command, shell=os.name == "nt", cwd=repo_root, env=env,
            text=True, capture_output=True,
        )
        if result.returncode != 0:
            print(result.stdout)
            print(result.stderr)
            raise RuntimeError(f"Command failed with exit code {result.returncode}.")
        return result

    # Child processes write to the kernel console, not the cell, so stream their output into the cell.
    process = subprocess.Popen(
        popen_command, shell=os.name == "nt", cwd=repo_root, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors="replace", bufsize=1,
    )
    lines = []
    for line in process.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f"Command failed with exit code {returncode}.")
    return subprocess.CompletedProcess(command, returncode, stdout="".join(lines))

def run_json(command):
    result = run(command, capture_output=True)
    return json.loads(result.stdout)

print(f"Repository: {repo_root}")
print(f"Lab: {lab_dir}")

## 2. Check prerequisites, create the resource group, and validate

This verifies Azure CLI authentication, selects the configured subscription, checks Bicep, creates the resource group, and validates the template against the subscription before anything is provisioned.

**This cell creates an empty Azure resource group, but nothing billable.**

In [ ]:
# Prerequisites
run(["az", "account", "show", "--output", "table"])
if subscription_id.strip():
    run(["az", "account", "set", "--subscription", subscription_id])
    run(["az", "account", "show", "--output", "table"])
run(["az", "bicep", "version"])
print("Prerequisite checks passed. Confirm that the selected subscription has 15 PTUs for each PTU deployment (45 in total: 30 in eastus2 for backends 1 and 4, and 15 in swedencentral for backend 2) before deploying.")

# Resource group
run([
    "az", "group", "create",
    "--name", resource_group_name,
    "--location", resource_group_location,
    "--output", "table",
])

# Template validation
_ = run([
    "az", "deployment", "group", "validate",
    "--resource-group", resource_group_name,
    "--name", deployment_name,
    "--template-file", template_file,
    "--parameters", parameters_file,
    "--output", "table",
])

## 3. Deploy resources

**Cost-incurring action:** this creates APIM, monitoring resources, four Foundry resources, three 15-PTU deployments, and four standard deployments.

In [ ]:
_ = run([
    "az", "deployment", "group", "create",
    "--resource-group", resource_group_name,
    "--name", deployment_name,
    "--template-file", template_file,
    "--parameters", parameters_file,
    "--output", "table",
])

## 4. Send traffic

This loads the deployment details (the gateway URL and the APIM subscription key, which stays in kernel memory and is never printed) and then runs `invoke-lab.ps1`. Set `phases` in the cell to choose what runs: `"all"`, or any list such as `"1"`, `"2"`, `"3"`, `"4"` or `"1,3"`. Each phase works on its own, so you can demo them one at a time.

1. **Weighted baseline** (`weighted_request_count`): small concurrent requests that show the 53/47 split.
2. **Pressure burst** (`pressure_request_count` and `SmallRequestCount`): large unique prompts sent through APIM that saturate both 15-PTU deployments, so requests spill over natively, are retried on the other backend, and sometimes reach backend 3.
3. **Guaranteed traveler:** the script saturates both PTUs directly, like another consumer of the same Foundry resources, then sends one tracer request through APIM and repeats until the tracer walks backend 1, backend 2, backend 3 and is answered.
4. **Spillover demo on backend 4** (`spillover_rounds` and `spillover_requests_per_round`): backend 4 is a separate 15-PTU deployment with Foundry-native spillover, outside the pool. The script saturates its PTU directly, then sends small requests through APIM. Foundry rejects them at the PTU and serves them from the spillover deployment, and APIM makes a single attempt and never sees a 429. The requests are small, so the spillover quota is not used up.

Phases 3 and 4 call the Foundry accounts with your Azure CLI identity, so they need data-plane access (for example the `Cognitive Services OpenAI User` role) and a machine that can resolve the accounts' host names. Circuit breakers stay open for 30-60 seconds after a 429, so wait about a minute between separate phase 2 and phase 3 runs. Phase 4 does not touch the pool, so it can run at any time. A full run takes a few minutes.

In [ ]:
import re

# Load the deployment details
outputs = run_json([
    "az", "deployment", "group", "show",
    "--resource-group", resource_group_name,
    "--name", deployment_name,
    "--query", "properties.outputs",
    "--output", "json",
])

gateway_url = outputs["gatewayUrl"]["value"]
subscription_key = outputs["subscriptionKey"]["value"]
application_insights_name = outputs["applicationInsightsName"]["value"]
log_analytics_workspace_name = outputs["logAnalyticsWorkspaceName"]["value"]

print(f"Gateway URL: {gateway_url}")
print(f"Application Insights: {application_insights_name}")
print(f"Log Analytics workspace: {log_analytics_workspace_name}")
print("APIM subscription key loaded (value hidden).")
print()

#============= Traffic settings =============#

# Which phases to run: "all", or any list such as "1", "2", "3", "4" or "1,3". 
# 1 = weighted baseline 53/47, 2 = pressure burst, 3 = guaranteed traveler, 4 = backend 4: PTU -> Foundry spillover.
phases = "3"   

weighted_request_count = 1000   # Phase 1: small concurrent requests. A few hundred are needed to see the 53/47 split.
pressure_request_count = 16     # Phase 2: large unique prompts sent through APIM that saturate the 15-PTU deployments organically.
small_request_count = 24   # Phase 2: small requests spread across the first seconds of that burst
spillover_rounds = 3   # Phase 4: how many times to saturate backend 4's PTU
spillover_requests_per_round = 12   # Phase 4: small requests per round sent through APIM. Keep them small so the spillover quota is not used up.
prompt_repeat = 9000 # Phase 3: Prompt size is PromptRepeat x ~10 tokens.
traveler_cycles = 10  # Phase 3: how many times to saturate the PTUs and send a tracer, stopping as soon as the tracer
    # walks backend 1 -> 2 -> 3. The first hop is weighted, so a tracer can also walk 2 -> 1 -> 3
prime_requests_per_backend = 12  # Phase 3: number of prime requests to send per backend
prime_prompt_repeat = 20000  # Phase 3: Prompt size is PromptRepeat x ~10 tokens

invoke_environment = os.environ.copy()
invoke_environment["APIM_SUBSCRIPTION_KEY"] = subscription_key

# Phases 3 and 4 saturate PTU backends directly (backends 1 and 2 for phase 3, backend 4 for phase 4). Key-based auth may be disabled on the accounts, so use an Entra ID token.
foundry_accounts = run_json(["az", "cognitiveservices", "account", "list", "--resource-group", resource_group_name, "--output", "json"])
foundry_token = run_json(["az", "account", "get-access-token", "--resource", "https://cognitiveservices.azure.com", "--output", "json"])["accessToken"]
prime_targets = [
    {"name": re.sub(r"-[a-z0-9]+$", "", account["name"]), "endpoint": account["properties"]["endpoint"], "token": foundry_token}
    for account in sorted(foundry_accounts, key=lambda account: account["name"])
    if re.match(r"route-backend-[124]-", account["name"])
]
invoke_environment["FOUNDRY_PRIME_TARGETS"] = json.dumps(prime_targets)

invoke_command = [
    powershell,
    "-NoProfile",
    "-File", invoke_script,
    "-GatewayUrl", gateway_url,
    "-Phase", phases,
    "-WeightedRequestCount", str(weighted_request_count),
    "-PressureRequestCount", str(pressure_request_count),
    "-SmallRequestCount", str(small_request_count),
    "-SpilloverRounds", str(spillover_rounds),
    "-SpilloverRequestsPerRound", str(spillover_requests_per_round),
    "-PromptRepeat", str(prompt_repeat),
    "-TravelerCycles", str(traveler_cycles),
    "-PrimeRequestsPerBackend", str(prime_requests_per_backend),
    "-PrimePromptRepeat", str(prime_prompt_repeat)
]
invoke_result = run(invoke_command, env=invoke_environment)

match = re.search(r"Traveler request ID: ((?:pressure|small|tracer)-[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12})", invoke_result.stdout)
traveler_request_id = match.group(1) if match else None
match = re.search(r"Spillover request ID: (spill-[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12})", invoke_result.stdout)
spillover_request_id = match.group(1) if match else None
match = re.search(r"Rejected request ID: ((?:pressure|small|tracer)-[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12})", invoke_result.stdout)
rejected_request_id = match.group(1) if match else None
selected_phases = {"1", "2", "3", "4"} if phases.strip().lower() == "all" else set(re.split(r"[,\s]+", phases.strip()))

print()
if traveler_request_id:
    print(f"Traveler request ID: {traveler_request_id}")
    print(f"Run telemetry.kql in the Log Analytics workspace {log_analytics_workspace_name}, pasting this ID as requestId in queries 1 and 5.")
elif selected_phases & {"2", "3"}:
    print("No request used all three backends. Wait about a minute for the circuit breakers to reset, then run this cell again.")
if rejected_request_id:
    print(f"Rejected request ID (used all its attempts but was rejected with a 429): {rejected_request_id}")
    print(f"Run telemetry.kql in the Log Analytics workspace {log_analytics_workspace_name}, pasting this ID as requestId in queries 1 and 5, to see each backend it tried.")
if spillover_request_id:
    print(f"Spillover request ID: {spillover_request_id}")
    print(f"Run telemetry-spillover.kql in the Log Analytics workspace {log_analytics_workspace_name}. It needs no ID: it reads every request whose ID starts with spill-.")
elif "4" in selected_phases:
    print("Phase 4 produced no spillover request. Read the messages above, then run this cell again.")
if selected_phases <= {"1"}:
    print(f"Only phase 1 ran, so there is no traveler or spillover request. Run query 2 of telemetry.kql in the Log Analytics workspace {log_analytics_workspace_name} for the 53/47 split.")

## 5. Inspect telemetry

1. In the Azure portal, open the **Log Analytics workspace** printed above (`log_analytics_workspace_name`) and select **Logs**. Use the workspace, not the Application Insights resource: the queries use the workspace table names `AppTraces` and `AppDependencies`, while the Application Insights Logs blade uses the older names `traces` and `dependencies`.
2. Open `telemetry.kql` from this lab folder and run one query at a time.
3. Replace `REPLACE-ME` with the traveler request ID printed by the previous cell (also stored in `traveler_request_id`).
4. Run query 1 to see that request go backend 1, backend 2, backend 3. Run query 2 for the 53/47 split, query 3 for every path the burst took, and query 4 for native spillover.

**For the phase 4 spillover demo,** open `telemetry-spillover.kql` instead and run its queries in order. It needs no request ID. Query 0 is a one-row scorecard, and queries 1, 2 and 3 answer the three questions: the PTU was exhausted, the requests were served by the spillover deployment, and APIM did nothing because Foundry handled it.

Telemetry ingestion can take several minutes.

## 6. Clean up

**Destructive and irreversible:** this first deletes the model deployments in each Foundry account (the resource group deletion does not succeed while they exist), then permanently deletes the entire resource group and waits until Azure confirms deletion. It then purges the soft-deleted API Management service and Foundry accounts from that resource group, so the same names can be redeployed. It also purges any left over from an earlier deletion of the same resource group.

Return to the configuration cell in section 1, set `cleanup_confirm` to the exact value of `resource_group_name`, rerun that cell, and then run the cleanup cell.

In [ ]:
if cleanup_confirm != resource_group_name:
    raise RuntimeError(
        "Cleanup blocked. Set cleanup_confirm to the exact resource_group_name in the configuration cell."
    )

def normalize_location(location):
    return location.replace(" ", "").lower()

def in_resource_group(resource_id):
    return f"/resourcegroups/{resource_group_name}/".lower() in resource_id.lower()

resource_group_exists = run_json(["az", "group", "exists", "--name", resource_group_name])

# 1. Delete the model deployments in each Foundry account. The resource group deletion does not succeed while they exist.
#    Provisioned (PTU) deployments go first because each one references its spillover deployment.
if resource_group_exists:
    for account in run_json(["az", "cognitiveservices", "account", "list", "--resource-group", resource_group_name, "--output", "json"]):
        deployments = run_json(["az", "cognitiveservices", "account", "deployment", "list", "-g", resource_group_name, "-n", account["name"], "--output", "json"])
        for deployment in sorted(deployments, key=lambda deployment: "Provisioned" not in deployment["sku"]["name"]):
            run(["az", "cognitiveservices", "account", "deployment", "delete", "-g", resource_group_name, "-n", account["name"], "--deployment-name", deployment["name"]])

# 2. Delete the resource group and wait until Azure confirms the deletion.
if resource_group_exists:
    run([
        "az", "group", "delete",
        "--name", resource_group_name,
        "--yes",
    ])
    print(f"Azure confirmed deletion of resource group: {resource_group_name}")
else:
    print(f"Resource group {resource_group_name} does not exist. Checking for soft-deleted resources.")

# 3. Purge the soft-deleted API Management service and Foundry accounts so the same names can be redeployed.
#    This also purges any left over from an earlier deletion of this resource group.
deleted_apim_services = [
    service for service in run_json(["az", "apim", "deletedservice", "list", "--output", "json"])
    if in_resource_group(service["serviceId"])
]
for service in deleted_apim_services:
    run(["az", "apim", "deletedservice", "purge", "--service-name", service["name"], "--location", normalize_location(service["location"])])

deleted_foundry_accounts = [
    account for account in run_json(["az", "cognitiveservices", "account", "list-deleted", "--output", "json"])
    if in_resource_group(account["id"])
]
for account in deleted_foundry_accounts:
    run(["az", "cognitiveservices", "account", "purge", "-g", resource_group_name, "-n", account["name"], "-l", normalize_location(account["location"])])

print(f"Purged {len(deleted_apim_services)} API Management service(s) and {len(deleted_foundry_accounts)} Foundry account(s). The same names can be redeployed.")